In [23]:
### 
import numpy as np
from src.op_utils import *
from src.gf2_utils import *
from src.sym import find_approx_symm
from openfermion import count_qubits, jordan_wigner, QubitOperator

n_H = 8
n_qubits = 2*n_H

H, mol =  build_H_chain_for_R(1.0, n_H)
n_qubits = 2*n_H

sym_ops_all, eps = find_approx_symm(jordan_wigner(H), n_H)

Added Pauli string: Z2 Z3 Z6 Z7 Z10 Z11 Z14 Z15 at threshold: 0.0
Added Pauli string: Z1 Z3 Z5 Z7 Z9 Z11 Z13 Z15 at threshold: 0.0
Added Pauli string: Z0 Z3 Z4 Z7 Z8 Z11 Z12 Z15 at threshold: 0.0
Added Pauli string: Z9 Z11 Z13 Z15 at threshold: 0.023505201991684058
Added Pauli string: Z8 Z10 Z12 Z14 at threshold: 0.023505201991684058
Added Pauli string: Z12 Z13 Z14 Z15 at threshold: 0.02820624239002087
Added Pauli string: Z11 Z13 Z15 at threshold: 0.02820624239002087
Added Pauli string: Z10 Z13 Z15 at threshold: 0.02820624239002087
Added Pauli string: Z7 at threshold: 0.02820624239002087
Added Pauli string: Z6 at threshold: 0.02820624239002087
Added Pauli string: Z4 Z5 at threshold: 0.02820624239002087
Added Pauli string: Z13 Z15 at threshold: 0.03290728278835768
Added Pauli string: Z5 at threshold: 0.03290728278835768
Added Pauli string: Z14 Z15 at threshold: 0.03760832318669449
Added Pauli string: Z3 at threshold: 0.03760832318669449
Added Pauli string: Z15 at threshold: 0.0423093635

In [14]:
mol.fci_energy

np.float64(-3.236066279892343)

In [4]:
# study correlation between l1 norm/expectation values of symmetry commutator with the number of subspaces/l1 norm of the overlaps. more subspaces/uniformly distributed, higher l1 norm.
from itertools import product
from openfermion import commutator, get_sparse_operator, expectation, get_ground_state, hermitian_conjugated

def construct_projectors(sym_list: list[QubitOperator]):
    """
    Construct projectors to all subspaces defined by Pauli symmetries sym_list

    """
    if len(sym_list) == 0:
        return [QubitOperator('', coefficient=1.0)]
    
    projectors = []

    sym = sym_list[0]
    projectors_rec = construct_projectors(sym_list=sym_list[1:])
    for proj in projectors_rec:
        projectors.append((0.5 + 0.5 * sym)*proj)
        projectors.append((0.5 - 0.5 * sym)*proj)
    return projectors

def l1norm(op: QubitOperator):
    return np.sum(np.abs(list(op.terms.values())))

def uni_grading(sym_ops, H):
    return sum([l1norm(commutator(sym, H)) for sym in sym_ops])

def state_grading(sym_ops, H, state, n_qubits):
    commutators = [commutator(sym, H) for sym in sym_ops]

    return np.sum([expectation(get_sparse_operator(hermitian_conjugated(comm) * comm, n_qubits), state) for comm in commutators])

def find_overlaps_l1(sym_ops, state, n_qubits):
    projs = construct_projectors(sym_ops)

    coeffs = np.sqrt([expectation(get_sparse_operator(proj, n_qubits), state) for proj in projs])
    return np.sum(np.abs(coeffs)), coeffs

sym_ops =  sym_ops_all[4:]

fci_e, fci = get_ground_state(get_sparse_operator(H, n_qubits))
uni_hct = uni_grading(sym_ops, jordan_wigner(H))
state_hct = state_grading(sym_ops, jordan_wigner(H), fci, n_qubits)

l1_hct, c = find_overlaps_l1(sym_ops, fci, n_qubits)

print(uni_hct, state_hct, l1_hct)

10.807957856918472 (0.9356597042557485+1.8214596497756474e-17j) 1.6528448137122997


In [17]:
sym_ops =  sym_ops_all[:n_qubits//2]
projs = construct_projectors(sym_ops)
projs_sparse = [get_sparse_operator(proj, n_qubits) for proj in projs]
Hs = get_sparse_operator(jordan_wigner(H), n_qubits)
min([get_ground_state(proj @ Hs @ proj)[0] for proj in projs_sparse])

np.float64(-3.1418504961498717)

In [19]:
sym_ops

[1.0 [Z2 Z3 Z6 Z7 Z10 Z11],
 1.0 [Z1 Z3 Z5 Z7 Z9 Z11],
 1.0 [Z0 Z3 Z4 Z7 Z8 Z11],
 1.0 [Z6 Z7 Z8 Z9 Z10 Z11],
 1.0 [Z7 Z9 Z11],
 1.0 [Z4 Z5 Z8 Z9 Z10 Z11]]

In [18]:
sen_syms = [QubitOperator('Z{} Z{}'.format(2*i, 2*i+1), 1) for i in range(n_H)]
projs = construct_projectors(sen_syms)
projs_sparse = [get_sparse_operator(proj, n_qubits) for proj in projs]
Hs = get_sparse_operator(jordan_wigner(H), n_qubits)
min([get_ground_state(proj @ Hs @ proj)[0] for proj in projs_sparse])

np.float64(-3.170861321033622)

In [10]:
projs = construct_projectors(sym_ops_all[:4])
l1_hct, c = find_overlaps_l1(sym_ops_all[:4], fci, n_qubits)
print(l1_hct)
min([get_ground_state(get_sparse_operator(proj * jordan_wigner(H) * proj, n_qubits))[0] for proj in projs])

1.0120323563836673


np.float64(-2.1662150951072685)

In [ ]:
sen_syms = [QubitOperator('Z{} Z{}'.format(2*i, 2*i+1), 1) for i in range(n_H)]
l1_sen, c2 = find_overlaps_l1(sen_syms, fci, n_qubits) #actual goodness
uni_sen = uni_grading(sen_syms, jordan_wigner(H)) #proxy
state_sen = state_grading(sen_syms, jordan_wigner(H), fci, n_qubits) #better proxy
#to check of proxy is positively correlated to actual goodness of symmetries

#doesn't seem very well correlated (test with sym_ops_all[3:7])

print(uni_sen, state_sen, l1_sen)
#np.sort(c2)

11.59420492739591 (1.0572039413412453-1.0408340855860843e-17j) 1.1452346603817158
